In [1]:
# Промпты
SYSTEM = (
    "You are a specialized analyst for government organizations. "
    "Analyze budgets by KOSGU, procurement according to 44-FZ, "
    "develop development strategies for budget directions. "
    "Respond in Russian with structured analysis."
)

prompt = "Проанализируй бюджет на закупку оборудования (КОСГУ 310) 10 млн руб. Риски по 44-ФЗ при сокращении 15%?"

In [2]:
"""
Инференс обученной модели GovAI Qwen3.5-4B с поддержкой стриминга
"""

!pip install -q unsloth
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
import torch
import threading

LORA_OUTPUT = "/kaggle/input/models/startexe/govai-qwen-lora-v11/other/default/1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT,
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def stream_generate(prompt: str, max_new_tokens: int = 1024):
    """Генерация с потоковым выводом токенов."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    attention_mask = torch.ones_like(inputs)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    # Генерация в отдельном потоке (streamer блокирует до первого токена)
    generation_kwargs = dict(
        input_ids=inputs,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=1.0,
        top_k=20,
        top_p=0.95,
        min_p=0.0,
        repetition_penalty=1.0,
        do_sample=True,
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for text in streamer:
        yield text

    thread.join()


if __name__ == "__main__":
    print("Streaming output:")
    print("-" * 40)
    for token_text in stream_generate(prompt, max_new_tokens=4096):
        print(token_text, end="", flush=True)
    print("\n" + "-" * 40)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 24.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 0:00

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Streaming output:
----------------------------------------
The user wants me to analyze a budget for equipment procurement (COSGU 310) of 10 million rubles. They're asking about risks according to 44-FZ if there's a 15% reduction.

I need to respond as a government organization specialist, using Russian, with structured analysis. Let me analyze this properly.

Key points to address:
- COSGU 310 = Equipment and software for main activities
- Budget: 10 million rubles
- Risk scenario: 15% reduction
- Framework: 44-FZ (Law on Procurement)

I should provide structured analysis with specific details about the applicable rules and risks.
</think>

## Бюджет на закупку оборудования

**КОСГУ:** 310
**Бюджет:** 10 000 000 руб.
**Риск:** сокращение 15%

### Распределение по КОСГУ

- 310 — Оборудование, инструменты, технологии (основные) — 6-8 млн руб.
- 320 — Услуги по управлению и поддержке оборудования — до 2-3 млн руб.

### Распорядитель: Администрация

### Законодательная база

- 44-ФЗ ст. 3

In [4]:
"""
Инференс НЕобученной модели Qwen3.5-4B с поддержкой стриминга
"""
!pip install -q unsloth
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
import torch
import threading

# БАЗОВАЯ модель, без LoRA
BASE_MODEL = "unsloth/Qwen3.5-4B"  # <-- базовая, без адаптера

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def stream_generate(prompt: str, max_new_tokens: int = 1024):
    """Генерация с потоковым выводом токенов."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user", "content": [{"type": "text", "text": prompt}]},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    attention_mask = torch.ones_like(inputs)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    generation_kwargs = dict(
        input_ids=inputs,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=1.0,
        top_k=20,
        top_p=0.95,
        min_p=0.0,
        repetition_penalty=1.0,
        do_sample=True,
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for text in streamer:
        yield text

    thread.join()


if __name__ == "__main__":
    print("=" * 60)
    print("БАЗОВАЯ МОДЕЛЬ (без LoRA) — Streaming output:")
    print("-" * 40)
    for token_text in stream_generate(prompt, max_new_tokens=4096):
        print(token_text, end="", flush=True)
    print("\n" + "-" * 40)

==((====))==  Unsloth 2026.6.7: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

БАЗОВАЯ МОДЕЛЬ (без LoRA) — Streaming output:
----------------------------------------
好的，用户让我分析预算中1000万卢布的采购设备预算，特别是KOSGU 310，以及根据44-FZ法在减少15%时的风险。首先，我需要理解KOSGU 310是什么。KOSGU是俄罗斯联邦的预算科目代码，310属于信息通信类设备，比如计算机、网络设备之类的。预算金额1000万，需要确定采购的具体内容，比如可能用于什么部门，是否有指定供应商。

接下来，用户提到44-FZ风险，这是关于联邦法律下的采购，减少预算15%可能有哪些影响。根据44-FZ，预算调整需要遵循一定程序，比如需要正当理由，避免歧视性影响，还要确保不影响项目执行。减少预算可能导致无法按原计划采购，或者需要重新招标，影响项目进度。

需要检查是否有合规问题，比如是否满足采购的最低要求，是否有足够预算覆盖需求。可能风险包括法律程序延迟、供应商流失、影响服务连续性。另外，用户可能希望知道如何缓解这些风险，比如提前与供应商沟通，准备备用预算方案，或者寻找替代资金来源。

还要考虑用户身份，可能是政府机构的采购负责人，需要合规分析，所以回答必须准确引用44-FZ条款，避免法律风险。结构上应该分点说明风险、合规检查、建议措施，确保用户能清楚了解情况并采取措施。
</think>

### Бюджет на закупку оборудования (КОСГУ 310) и анализ рисков при сокращении по 44-ФЗ

#### **1. Структура и обоснование расходов (КОСГУ 310)**
**КОСГУ 310** относится к закупке оборудования для информационных и телекоммуникационных систем. При объёме в 10 млн руб. необходимо уточнить:
- **Назначение оборудования** (например, серверное оборудование, сетевые системы, ПО для обработки да